# Portfolio Manager

Clean pipeline for mutual fund portfolio construction.

**Pipeline:**
1. Environment Setup
2. Paths & Parameters
3. Utility Functions
4. Core Logic (Screen → Rank → Overlap → Optimize)
5. Load & Validate Data
6. Resolve Fund Names
7. Run Pipeline
8. Results

**Design principles:**
- Input fund list is already shortlisted — no fund is hard-dropped. Missing data excludes that component from risk_score (weights renormalized over available subset).
- Hybrid score = `abs_score` (pre-tested, upstream) + `risk_score` (peer-relative, computed here).
- `risk_score` components: Sharpe 3Y (35%) + Sortino 3Y (30%) + Sharpe 5Y (35%). Expense ratio and AUM excluded — not present in source data.
- `risk_score` measures **capital efficiency** (higher = better Sharpe/Sortino vs peers = lower volatility per unit return). Conservative profiles weight it higher; aggressive profiles weight abs_score (raw long-term track record) higher.
- Profile → weights: `conservative` (abs 35% / risk 65%), `moderate` (50/50), `aggressive` (abs 65% / risk 35%).
- Ratio scores are **peer-relative min-max normalized within the universe**: raw ratio = fund/category_avg (preserving category-relativity), then min-max to [0,100] across all funds.
- Both abs_score and risk_score normalized to [0, 100] within the universe before blending — so the profile split is genuine.
- Overlap uses Sørensen (average) normalization, not min.
- Optimizer relaxes overlap constraint gradually rather than silently falling back to top-N.
- Per-fund weight floor and cap applied after score-proportional allocation.


## 1. Environment Setup

In [15]:
%load_ext autoreload
%autoreload 2

import logging
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations
from math import comb


np.random.seed(42)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

logging.basicConfig(level=logging.INFO, format='%(levelname)s — %(message)s')
logger = logging.getLogger(__name__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Paths & Parameters

Edit this cell only — no need to touch logic cells.

In [16]:
# ── Paths ──────────────────────────────────────────────────────────────────────
RAW_FUNDS_PATH   = Path('../mutualfunds/raw_funds.tsv')
FUND_INFO_DIR    = Path('../mutualfunds/fund_info')
FUND_SCORES_PATH = Path('../mutualfunds/fund_scores.tsv')  # precomputed by fund_selection_strategy.ipynb
OUTPUT_DIR       = Path('tuning_results/portfolio_debug')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Portfolio Parameters ───────────────────────────────────────────────────────
NUM_FUNDS    = 8
RISK_PROFILE = 'aggressive'   # conservative | moderate | aggressive

# ── Profile → Hybrid Weight Mapping ───────────────────────────────────────────
# risk_score = Sharpe/Sortino efficiency (higher = less volatile, more capital-efficient)
# abs_score  = long-term return track record (10Y/5Y/3Y percentile + stability + Crisil)
#
# conservative: prioritise capital efficiency — higher risk_score weight
# moderate    : balanced
# aggressive  : prioritise raw long-term return track record — higher abs_score weight
_PROFILE_WEIGHTS = {
    'conservative': (0.35, 0.65),   # (W_ABS, W_RISK)
    'moderate':     (0.50, 0.50),
    'aggressive':   (0.65, 0.35),
}
W_ABS, W_RISK = _PROFILE_WEIGHTS[RISK_PROFILE]

# ── Risk Score Component Weights (must sum to 1.0) ────────────────────────────
# Sharpe 3Y  : risk-adjusted return, recent cycle
# Sortino 3Y : downside risk, recent cycle
# Sharpe 5Y  : medium-term consistency — counters 3Y procyclicality
# Note: Expense ratio and AUM were removed — data not available in source files.
W_SHARPE_3Y = 0.35
W_SORTINO   = 0.30
W_SHARPE_5Y = 0.35

assert abs(W_SHARPE_3Y + W_SORTINO + W_SHARPE_5Y - 1.0) < 1e-9, \
    'Risk score component weights must sum to 1.0'

# ── Weight Allocation Bounds ───────────────────────────────────────────────────
# Clipped and renormalized after score-proportional allocation.
# For NUM_FUNDS=8: floor=6.25%, cap=31.25%
WEIGHT_FLOOR = 1.0 / (2.0 * NUM_FUNDS)
WEIGHT_CAP   = 2.5 / NUM_FUNDS

# ── Overlap Constraint ─────────────────────────────────────────────────────────
MAX_OVERLAP_PCT = 40.0

# ── Fund Universe ──────────────────────────────────────────────────────────────
FUND_NAMES = [
    "HSBC Value Dir Gr",
    "Nippon India Large Cap Dir Gr",
    "Quant ELSS Tax Saver Dir Gr",
    "Nippon India Small Cap Dir Gr",
    "ICICI Pru Dividend Yield Eq Dir Gr",
    "ICICI Pru FMCG Dir Gr",
    "Invesco India Financial Serv Dir Gr",
    "Bandhan Large & Mid Cap Dir Gr",
    "ICICI Pru Focused Equity Dir Gr",
    "Edelweiss Mid Cap Dir Gr",
    "Parag Parikh Flexi Cap Dir Gr",
    "Quant Flexi Cap Dir Gr",
    "Quant ESG Integration Strat Dir Gr",
    "HDFC Flexi Cap Dir Gr",
    "ICICI Prudential Large Cap Dir Gr",
    "SBI Technology Opportunities Dir Gr",
    "HDFC Focused Dir Gr",
    "Bank of India Manfactrg &Infra Dir Gr",
    "Kotak India Growth Ser 4 Dir Gr",
    "Tata India Consumer Dir Gr",
    "DSP Nat Res & New Enrgy Dir Gr",
    "Nippon India Growth Mid Cap Dir Gr",
    "Sundaram LT Tax Ad Sr III Dir Gr",
    "Sundaram LT Tax Ad Sr IV Dir Gr",
    "Nippon India Multi Cap Dir Gr",
    "Mirae Asset Great Consumer Dir Gr",
    "Motilal Oswal Nifty Midcap 150 Idx DirGr",
    "HDFC Mid Cap Dir Gr",
    "Invesco India largecap Dir Gr",
    "SBI Banking & Financial Svcs Dir Gr",
    "Motilal Oswal ELSS Tax Saver Dir Gr",
    "Nippon India Nifty Midcap 150 Idx Dir Gr",
    "Invesco India Mid Cap Dir Gr",
    "Edelweiss MSCI India D&W HC 45 Dir Gr",
    "Invesco India PSU Equity Dir Gr",
    "Franklin Build India Dir Gr",
    "ICICI Pru Infrastructure Dir Gr",
    "Mahindra Manulife Multi Cap Dir Gr",
    "Kotak Contra Dir Gr",
    "ICICI Pru Large & Mid Cap Dir Gr",
    "Invesco India large& mid cap Dir Gr",
    "DSP India T.I.G.E.R. Dir Gr",
    "LIC MF Nifty Next 50 Index Dir Gr",
    "Bank of India Flexi Cap Fund Dir Gr",
    "Edelweiss Large Cap Dir Gr",
    "Kotak Large & Midcap Dir Gr",
    "DSP Nifty 50 Equal Weight Index Dir Gr",
    "ICICI Pru BHARAT 22 FOF Dir Gr",
    "ICICI Pru Nifty Next 50 Index Dir Gr",
    "Edelweiss Flexi Cap Dir Gr",
    "Kotak Focused Dir Gr",
    "Motilal Oswal Large & Midcap Dir Gr",
    "Nippon India Pharma Dir Gr",
    "Sundaram LT Mic cap Tax Ad Sr IV Dir Gr",
    "Motilal Oswal BSE Enh Val Idx Dir Gr",
    "SBI Focused Dir Gr",
    "Sundaram LT Mic capTax Ad Sr V Dir Gr",
    "Nippon India Power & Infra Dir Gr",
    "ICICI Pru Nifty Auto Index Dir Gr",
    "Aditya BSL PSU Equity Dir Gr",
    "Bank of India ELSS Tax Saver Dir Gr",
    "SBI PSU Dir Gr",
    "Motilal Oswal Nasdaq 100 Fd of Fd Dir Gr",
    "Mahindra Manulife Focused Dir Gr",
    "SBI ESG Exclusionary Strategy Dir Gr"
]

# ── Validate prerequisites ─────────────────────────────────────────────────────
assert RAW_FUNDS_PATH.exists(),   f'raw_funds.tsv not found at {RAW_FUNDS_PATH}'
assert FUND_INFO_DIR.exists(),    f'fund_info dir not found at {FUND_INFO_DIR}'
assert FUND_SCORES_PATH.exists(), (
    f'fund_scores.tsv not found at {FUND_SCORES_PATH}. '
    'Run fund_selection_strategy.ipynb first.'
)
assert RISK_PROFILE in _PROFILE_WEIGHTS, \
    f'Invalid RISK_PROFILE: "{RISK_PROFILE}". Must be conservative | moderate | aggressive.'

logger.info(
    f'Parameters OK — {len(FUND_NAMES)} funds | profile={RISK_PROFILE} | '
    f'W_ABS={W_ABS:.0%} W_RISK={W_RISK:.0%} | '
    f'n={NUM_FUNDS} | max_overlap={MAX_OVERLAP_PCT}% | '
    f'weight=[{WEIGHT_FLOOR*100:.1f}%, {WEIGHT_CAP*100:.1f}%]'
)


INFO — Parameters OK — 65 funds | profile=aggressive | W_ABS=65% W_RISK=35% | n=8 | max_overlap=40.0% | weight=[6.2%, 31.2%]


## 3. Utility Functions

In [17]:
def safe_float(x):
    """
    Convert x to float, stripping commas. Returns None on failure.
    Single definition used everywhere — no local overrides anywhere in this notebook.
    Callers must handle None explicitly; never silently default to 0.0.
    """
    try:
        return float(str(x).replace(',', ''))
    except (ValueError, TypeError):
        return None


def normalize_series(s: pd.Series) -> pd.Series:
    """
    Min-max normalize a Series to [0, 100].
    Returns 50.0 for all if all values are identical (neutral midpoint).
    """
    mn, mx = s.min(), s.max()
    if mx == mn:
        return pd.Series(50.0, index=s.index)
    return (s - mn) / (mx - mn) * 100.0


def load_risk_row(isin: str, fund_info_dir: Path):
    """
    Load first row of risk_metrics_{isin}.tsv.
    Returns the row as a Series, or None if file missing/empty.
    """
    f = fund_info_dir / f'risk_metrics_{isin}.tsv'
    if not f.exists():
        return None
    df = pd.read_csv(f, sep='\t')
    return df.iloc[0] if not df.empty else None


def load_fund_scores(fund_scores_path: Path) -> dict:
    """
    Load precomputed absolute scores from fund_selection_strategy.ipynb.
    Returns dict keyed by ISIN.
    """
    df = pd.read_csv(fund_scores_path, sep='\t')
    return df.set_index('isin').to_dict(orient='index')


def clip_and_renormalize_weights(weights: dict,
                                  floor: float,
                                  cap: float,
                                  max_iter: int = 50) -> dict:
    """
    Iteratively clip weights to [floor, cap] and renormalize until stable.
    Ensures no fund gets a negligible or dominant allocation.
    Converges in a small number of iterations for typical portfolio sizes.
    """
    n = len(weights)
    assert floor * n <= 1.0, f'Floor {floor:.3f} × {n} funds > 1.0 — infeasible.'
    assert cap  * n >= 1.0, f'Cap {cap:.3f} × {n} funds < 1.0 — infeasible.'

    w = dict(weights)
    for _ in range(max_iter):
        total   = sum(w.values())
        w       = {k: v / total for k, v in w.items()}
        clipped = {k: min(max(v, floor), cap) for k, v in w.items()}
        if clipped == w:
            break
        w = clipped

    total = sum(w.values())
    return {k: v / total for k, v in w.items()}


logger.info('Utility functions defined.')

INFO — Utility functions defined.


## 4. Core Logic

### 4.1 Data Screen

No fund is dropped. Logs metric coverage so data gaps are visible before scoring.

In [18]:
def screen_funds(fund_isins: list, fund_info_dir: Path) -> pd.DataFrame:
    """
    Soft data screen — logs coverage, drops nothing.

    For each fund, checks availability of every metric used in scoring.
    Funds with partial data proceed; missing components are excluded from
    risk_score (weights renormalized over available components only).
    Returns a DataFrame with one row per fund and boolean coverage columns
    so the analyst can spot gaps before trusting the output.
    """
    REQUIRED_FIELDS = [
        'sharpe_3y', 'sharpe_cat_avg_3y',
        'sortino_3y', 'sortino_cat_avg_3y',
        'sharpe_5y', 'sharpe_cat_avg_5y',
    ]

    rows = []
    for isin in fund_isins:
        row = load_risk_row(isin, fund_info_dir)
        entry = {'ISIN': isin, 'file_found': row is not None}
        for field in REQUIRED_FIELDS:
            entry[field] = (row is not None) and (safe_float(row.get(field)) is not None)
        rows.append(entry)

    df = pd.DataFrame(rows)

    no_file = df[~df['file_found']]['ISIN'].tolist()
    if no_file:
        logger.warning(f'No risk_metrics file for {len(no_file)} funds: {no_file}')

    for field in REQUIRED_FIELDS:
        gap = df[~df[field]]['ISIN'].tolist()
        if gap:
            logger.warning(f'  [{field}] missing for {len(gap)} funds — excluded from risk_score: {gap}')

    full = df[REQUIRED_FIELDS].all(axis=1).sum()
    logger.info(
        f'Data screen: {len(fund_isins)} funds total | '
        f'{full} full coverage | {len(fund_isins)-full} partial (missing components excluded)'
    )
    return df


logger.info('screen_funds defined.')


INFO — screen_funds defined.


### 4.2 Ranking

In [19]:
def _ratio_raw(val, avg, invert=False):
    """
    Raw ratio of val vs category avg. Returns None if data is missing.
    invert=True for expense ratio (cheaper than avg is better).
    """
    if val is None or avg is None or avg == 0:
        return None
    r = (avg / val) if invert else (val / avg)
    return max(0.0, r)


def _normalize_ratios(ratios: list) -> list:
    """
    Min-max normalize a list of raw ratio values (with Nones) to [0, 100].

    Why peer-relative instead of ratio*50 (capped at 2x)?
      The ratio*50 formula caps at 100 when any fund exceeds 2x its category
      average. Genuine outliers (e.g. Parag Parikh Sortino = 2.21x avg) hit
      the ceiling and become indistinguishable from a fund at exactly 2.0x.
      Peer-relative normalization maps the worst ratio to 0 and best to 100,
      preserving full differentiation across the observed universe spread.
      Category-relativity is preserved because ratios are still computed
      against each fund's own category average before normalization.

    Returns a list of the same length; None entries stay None.
    """
    valid = [(i, v) for i, v in enumerate(ratios) if v is not None]
    if not valid:
        return ratios
    indices, values = zip(*valid)
    mn, mx = min(values), max(values)
    result = list(ratios)
    for i, v in zip(indices, values):
        result[i] = 50.0 if mx == mn else (v - mn) / (mx - mn) * 100.0
    return result


def rank_funds(fund_isins: list,
               fund_info_dir: Path,
               fund_scores_path: Path,
               isin_to_name: dict,
               w_abs:  float = W_ABS,
               w_risk: float = W_RISK) -> pd.DataFrame:
    """
    Hybrid scoring: absolute quality (w_abs) + peer-relative risk (w_risk).

    abs_score (0-100):
        Precomputed upstream. Missing ISIN falls back to 50 (neutral).

    risk_score (0-100) — components:
        Sharpe 3Y  (W_SHARPE_3Y): risk-adjusted return, recent cycle
        Sortino 3Y (W_SORTINO)  : downside risk, recent cycle
        Sharpe 5Y  (W_SHARPE_5Y): medium-term consistency

        Raw ratio = fund/category_avg (preserving category-relativity), then
        peer-relative min-max normalized within the universe to [0, 100].

        Expense ratio and AUM excluded — columns not present in source data.

    Missing data: None for that component; risk_score renormalizes weights
        over available components only. If ALL missing, defaults to 50.

    Blending:
        abs_score and risk_score normalized to [0,100] within the universe.
    """
    abs_lookup = load_fund_scores(fund_scores_path)

    # ── Pass 1: collect raw ratios for all funds ──────────────────────────────
    raw_rows = []
    for isin in fund_isins:
        row       = load_risk_row(isin, fund_info_dir)
        fund_name = isin_to_name.get(isin, '')

        def get(field):
            return (lambda v: float(str(v).replace(',', '')) if v is not None else None)(
                row.get(field) if row is not None else None
            )

        raw_rows.append({
            'isin':      isin,
            'fund_name': fund_name,
            'r_s3y':     _ratio_raw(get('sharpe_3y'),  get('sharpe_cat_avg_3y')),
            'r_sort':    _ratio_raw(get('sortino_3y'), get('sortino_cat_avg_3y')),
            'r_s5y':     _ratio_raw(get('sharpe_5y'),  get('sharpe_cat_avg_5y')),
            # raw values for display
            'sharpe_3y':  get('sharpe_3y'),
            'sortino_3y': get('sortino_3y'),
            'sharpe_5y':  get('sharpe_5y'),
        })

    # ── Pass 2: peer-normalize each ratio column across universe ──────────────
    for col in ('r_s3y', 'r_sort', 'r_s5y'):
        normalized = _normalize_ratios([r[col] for r in raw_rows])
        for r, v in zip(raw_rows, normalized):
            r[col] = v

    # ── Pass 3: blend into risk_score ─────────────────────────────────────────
    rows = []
    for r, isin in zip(raw_rows, fund_isins):
        s3y_score  = r['r_s3y']
        sort_score = r['r_sort']
        s5y_score  = r['r_s5y']

        components = [
            (s3y_score,  W_SHARPE_3Y),
            (sort_score, W_SORTINO),
            (s5y_score,  W_SHARPE_5Y),
        ]
        available = [(score, w) for score, w in components if score is not None]
        if available:
            total_w    = sum(w for _, w in available)
            risk_score = sum(score * w for score, w in available) / total_w
        else:
            risk_score = 50.0

        info      = abs_lookup.get(isin, {})
        abs_score = info.get('abs_score', 50.0)

        rows.append({
            'ISIN':            isin,
            'abs_score':       round(abs_score,  2),
            'risk_score':      round(risk_score, 2),
            'sharpe_3y_score': round(s3y_score,  2) if s3y_score  is not None else None,
            'sortino_score':   round(sort_score,  2) if sort_score is not None else None,
            'sharpe_5y_score': round(s5y_score,  2) if s5y_score  is not None else None,
            'sharpe_3y':       r['sharpe_3y'],
            'sortino_3y':      r['sortino_3y'],
            'sharpe_5y':       r['sharpe_5y'],
            'tier':   info.get('tier', '?'),
            'CV':     info.get('CV'),
            's10Y':   info.get('s10Y'),
            's5Y':    info.get('s5Y'),
            's_stab': info.get('s_stab'),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df['abs_norm']  = normalize_series(df['abs_score'])
    df['risk_norm'] = normalize_series(df['risk_score'])
    df['score']     = (w_abs * df['abs_norm'] + w_risk * df['risk_norm']).round(2)

    return df.sort_values('score', ascending=False).reset_index(drop=True)


logger.info('rank_funds defined.')


INFO — rank_funds defined.


### 4.3 Overlap Matrix

In [20]:
def compute_overlap_matrix(ranked_df: pd.DataFrame, fund_info_dir: Path) -> pd.DataFrame:
    """
    Pairwise holding overlap between funds (%).

    Overlap(A, B) = sum(min(w_a, w_b)) / avg(total_A, total_B) * 100

    Denominator is the AVERAGE of both totals (Sørensen-style).
    The previous min() denominator biased overlap upward for small funds
    paired with large ones, unfairly penalising large-small combinations.

    Funds with missing holdings score 0% overlap (conservative: not penalised
    for missing data, but also contribute no diversification evidence).
    """
    isins    = ranked_df['ISIN'].tolist()
    holdings = {}
    missing  = []

    for isin in isins:
        f = fund_info_dir / f'holdings_{isin}.tsv'
        if not f.exists():
            holdings[isin] = {}
            missing.append(isin)
            continue
        df = pd.read_csv(f, sep='\t')
        if df.empty:
            holdings[isin] = {}
            missing.append(isin)
            continue
        df['weight'] = df['weight'].apply(safe_float).fillna(0.0)
        holdings[isin] = df.groupby('stock_name')['weight'].sum().to_dict()

    if missing:
        logger.warning(
            f'Holdings missing for {len(missing)} funds '
            f'(overlap treated as 0%): {missing}'
        )

    n       = len(isins)
    overlap = pd.DataFrame(0.0, index=isins, columns=isins)

    for i in range(n):
        for j in range(i, n):
            a, b = isins[i], isins[j]
            if i == j:
                overlap.loc[a, b] = 100.0
                continue
            stocks_a, stocks_b = holdings[a], holdings[b]
            all_stocks = set(stocks_a) | set(stocks_b)
            raw   = sum(min(stocks_a.get(s, 0.0), stocks_b.get(s, 0.0)) for s in all_stocks)
            denom = (sum(stocks_a.values()) + sum(stocks_b.values())) / 2.0
            pct   = (raw / denom * 100.0) if denom > 0 else 0.0
            overlap.loc[a, b] = pct
            overlap.loc[b, a] = pct

    return overlap


logger.info('compute_overlap_matrix defined.')

INFO — compute_overlap_matrix defined.


### 4.4 Portfolio Optimizer

In [21]:
def optimize_portfolio(ranked_df:      pd.DataFrame,
                       overlap_matrix:  pd.DataFrame,
                       num_funds:       int   = NUM_FUNDS,
                       max_overlap:     float = MAX_OVERLAP_PCT,
                       weight_floor:    float = WEIGHT_FLOOR,
                       weight_cap:      float = WEIGHT_CAP) -> dict:
    """
    Combination-based portfolio optimizer.

    Selection:
        Evaluates all C(n, num_funds) combinations.
        Rejects any where any pairwise overlap > max_overlap.
        Selects the combo with the highest total hybrid score.

    Overlap fallback:
        If no combo passes, threshold relaxes in 5% steps up to 2x max_overlap,
        logged at each step. Raises ValueError if still no valid portfolio found.
        Replaces the previous silent top-N fallback.

    Weight allocation:
        Proportional to hybrid score — same objective as selection.
        Iteratively clipped to [weight_floor, weight_cap] and renormalized.
        No fund gets a negligible (<floor) or dominant (>cap) allocation.
    """
    isins     = ranked_df['ISIN'].tolist()
    score_map = dict(zip(ranked_df['ISIN'], ranked_df['score']))
    k         = min(num_funds, len(isins))

    def _find_best(threshold):
        best_combo, best_score = None, -float('inf')

        total = comb(len(isins), k)
        checked = 0
        step = max(1, total // 50)  # print every 2%

        for combo in combinations(isins, k):
            checked += 1

            # 👇 lightweight progress
            if checked % step == 0:
                logger.info(f'Progress: {checked}/{total} ({checked/total:.1%}) | threshold={threshold}%')

            if any(
                overlap_matrix.loc[combo[i], combo[j]] > threshold
                for i in range(len(combo))
                for j in range(i + 1, len(combo))
            ):
                continue

            total_score = sum(score_map[f] for f in combo)
            if total_score > best_score:
                best_score, best_combo = total_score, combo

        return best_combo, best_score
    best_combo, best_score = _find_best(max_overlap)
    effective_threshold    = max_overlap

    if best_combo is None:
        for relaxed in range(int(max_overlap) + 5, int(max_overlap * 2) + 1, 5):
            logger.warning(
                f'No valid portfolio at overlap ≤ {effective_threshold:.0f}%. '
                f'Relaxing to {relaxed}%.'
            )
            best_combo, best_score = _find_best(relaxed)
            effective_threshold    = relaxed
            if best_combo is not None:
                break

    if best_combo is None:
        raise ValueError(
            f'No valid portfolio found even at {effective_threshold:.0f}% overlap. '
            'Expand the fund universe or raise MAX_OVERLAP_PCT.'
        )

    raw   = {isin: max(score_map.get(isin, 0.0), 0.0) for isin in best_combo}
    total = sum(raw.values())
    initial = (
        {k: v / total for k, v in raw.items()}
        if total > 0
        else {isin: 1.0 / len(best_combo) for isin in best_combo}
    )
    weights = clip_and_renormalize_weights(initial, weight_floor, weight_cap)

    return {
        'selected_funds':        list(best_combo),
        'weights':               weights,
        'portfolio_score':       best_score,
        'effective_overlap_pct': effective_threshold,
    }


logger.info('optimize_portfolio defined.')

INFO — optimize_portfolio defined.


## 5. Load & Validate Data

In [22]:
raw_df = pd.read_csv(RAW_FUNDS_PATH, sep='\t')
raw_df.columns = [c.strip() for c in raw_df.columns]

assert 'schemeName' in raw_df.columns, 'Missing schemeName column in raw_funds.tsv'
assert 'isin'       in raw_df.columns, 'Missing isin column in raw_funds.tsv'

ISIN_TO_NAME = dict(zip(raw_df['isin'], raw_df['schemeName']))

def get_fund_name(isin: str) -> str:
    return ISIN_TO_NAME.get(isin, f'Unknown ({isin})')

logger.info(f'Loaded {len(raw_df)} records from raw_funds.tsv')

INFO — Loaded 924 records from raw_funds.tsv


## 6. Resolve Fund Names → ISINs

In [23]:
def resolve_fund_names(names: list, raw_df: pd.DataFrame) -> pd.DataFrame:
    """Exact case-insensitive match of fund names to ISINs."""
    rows = []
    for name in names:
        match = raw_df[raw_df['schemeName'].str.lower() == name.lower()]
        isin  = str(match.iloc[0]['isin']) if not match.empty else None
        rows.append({'fund_name': name, 'isin': isin, 'resolved': isin is not None})
    return pd.DataFrame(rows)


resolve_df = resolve_fund_names(FUND_NAMES, raw_df)
unresolved = resolve_df[~resolve_df['resolved']]

if not unresolved.empty:
    logger.warning(f'{len(unresolved)} fund names could not be resolved to ISINs:')
    print(unresolved[['fund_name']].to_string(index=False))

fund_isins = resolve_df.loc[resolve_df['resolved'], 'isin'].tolist()
assert fund_isins, 'No funds resolved. Check FUND_NAMES against raw_funds.tsv.'

logger.info(f'Resolved {len(fund_isins)}/{len(FUND_NAMES)} fund names to ISINs')

INFO — Resolved 65/65 fund names to ISINs


## 6.5 Download Missing Fund Data

Download `risk_metrics` and `holdings` for any fund whose files don't yet exist.
Calls `scripts/fetch_fund_information.py` via subprocess, piping the fund name via stdin.
Skips funds that already have both files.


In [24]:
import subprocess
import sys

# Script is relative to the workspace root, one level up from portfolio_manager/
FETCH_SCRIPT = Path('scripts/fetch_fund_information.py')

# Build a mapping from ISIN → fund name (as it appears in FUND_NAMES)
# so we can pass the exact name string the script was given.
isin_to_input_name = dict(zip(resolve_df['isin'], resolve_df['fund_name']))

missing = []
for isin in fund_isins:
    risk_file     = FUND_INFO_DIR / f'risk_metrics_{isin}.tsv'
    holdings_file = FUND_INFO_DIR / f'holdings_{isin}.tsv'
    if not risk_file.exists() or not holdings_file.exists():
        missing.append(isin)

if not missing:
    logger.info('All fund data files already exist — nothing to download.')
else:
    logger.info(f'{len(missing)} fund(s) need data download: {missing}')
    for isin in missing:
        fund_name = isin_to_input_name.get(isin, isin)
        logger.info(f'Fetching: {fund_name} ({isin}) …')
        result = subprocess.run(
            [sys.executable, str(FETCH_SCRIPT)],
            input=fund_name,
            text=True,
            capture_output=True,
            cwd=Path('../').resolve(),   # script resolves paths relative to workspace root
        )
        if result.returncode != 0 or '❌' in result.stdout:
            logger.warning(f'  FAILED for {fund_name}:\n{result.stdout.strip()}\n{result.stderr.strip()}')
        else:
            logger.info(f'  OK — {result.stdout.strip().splitlines()[-1]}')


INFO — All fund data files already exist — nothing to download.


## 7. Run Pipeline

In [25]:
# ── Step 1: Data Screen ───────────────────────────────────────────────────────
print('=' * 70)
print('STEP 1 — Data Screen (no funds dropped)')
print('=' * 70)

screen_df = screen_funds(fund_isins, FUND_INFO_DIR)
print(screen_df.to_string(index=False))

INFO — Data screen: 65 funds total | 65 full coverage | 0 partial (missing components excluded)


STEP 1 — Data Screen (no funds dropped)
        ISIN  file_found  sharpe_3y  sharpe_cat_avg_3y  sortino_3y  sortino_cat_avg_3y  sharpe_5y  sharpe_cat_avg_5y
INF917K01HD4        True       True               True        True                True       True               True
INF204K01XI3        True       True               True        True                True       True               True
INF966L01986        True       True               True        True                True       True               True
INF204K01K15        True       True               True        True                True       True               True
INF109KA1UA0        True       True               True        True                True       True               True
INF109K01Z14        True       True               True        True                True       True               True
INF205K01KY4        True       True               True        True                True       True               True
INF194K01V89        True

In [26]:
# ── Step 2: Rank Funds ────────────────────────────────────────────────────────
print('=' * 70)
print('STEP 2 — Ranking (hybrid = abs + risk)')
print('=' * 70)

ranked_df = rank_funds(
    fund_isins, FUND_INFO_DIR, FUND_SCORES_PATH, ISIN_TO_NAME,
    w_abs=W_ABS, w_risk=W_RISK
)
assert not ranked_df.empty, 'Ranking returned no results. Check risk_metrics files.'
ranked_df['name'] = ranked_df['ISIN'].map(get_fund_name)

print('\n── Hybrid score ─────────────────────────────────────────────────────')
print(ranked_df[[
    'ISIN', 'score', 'abs_norm', 'risk_norm', 'abs_score', 'risk_score', 'tier', 'name'
]].to_string(index=True))

print('\n── Risk score components ────────────────────────────────────────────')
print(ranked_df[[
    'ISIN',
    'sharpe_3y_score', 'sortino_score', 'sharpe_5y_score',
    'sharpe_3y', 'sortino_3y', 'sharpe_5y',
    'name'
]].to_string(index=True))

print('\n── Absolute score metadata ──────────────────────────────────────────')
print(ranked_df[['ISIN', 'abs_score', 'tier', 'CV', 's10Y', 's5Y', 's_stab', 'name']].to_string(index=True))


STEP 2 — Ranking (hybrid = abs + risk)

── Hybrid score ─────────────────────────────────────────────────────
            ISIN  score    abs_norm  risk_norm  abs_score  risk_score tier                                      name
0   INF204K01XI3  84.17   93.099069      67.58     101.99       67.58    A             Nippon India Large Cap Dir Gr
1   INF917K01HD4  80.76  100.000000      45.03     103.62       45.03    A                         HSBC Value Dir Gr
2   INF200K01RV6  71.99   56.900931     100.00      93.44      100.00    A       SBI Technology Opportunities Dir Gr
3   INF205K01KY4  70.08   69.305673      71.52      96.37       71.52    A       Invesco India Financial Serv Dir Gr
4   INF966L01986  68.87   80.567316      47.15      99.03       47.15    A               Quant ELSS Tax Saver Dir Gr
5   INF109KA1UA0  68.45   74.089754      57.97      97.50       57.97    A        ICICI Pru Dividend Yield Eq Dir Gr
6   INF109K018N2  66.35   66.807790      65.51      95.78       65.51  

In [27]:
# ── Step 3: Overlap Matrix ────────────────────────────────────────────────────
print('=' * 70)
print('STEP 3 — Overlap Matrix (Sørensen normalization)')
print('=' * 70)

overlap_df    = compute_overlap_matrix(ranked_df, FUND_INFO_DIR)
name_map      = {isin: get_fund_name(isin)[:22] for isin in overlap_df.index}
overlap_named = overlap_df.rename(index=name_map, columns=name_map)
print(overlap_named.round(1).to_string())

isins_list = overlap_df.index.tolist()
high_pairs = [
    (get_fund_name(isins_list[i])[:30],
     get_fund_name(isins_list[j])[:30],
     round(overlap_df.iloc[i, j], 1))
    for i in range(len(isins_list))
    for j in range(i + 1, len(isins_list))
    if overlap_df.iloc[i, j] > MAX_OVERLAP_PCT
]
if high_pairs:
    print(f'\n⚠️  Pairs above {MAX_OVERLAP_PCT:.0f}%:')
    for a, b, pct in sorted(high_pairs, key=lambda x: -x[2]):
        print(f'  {pct:5.1f}%  {a}  ↔  {b}')
else:
    print(f'\n✓ No pairs exceed {MAX_OVERLAP_PCT:.0f}%.')

STEP 3 — Overlap Matrix (Sørensen normalization)
                        Nippon India Large Cap  HSBC Value Dir Gr  SBI Technology Opportu  Invesco India Financia  Quant ELSS Tax Saver D  ICICI Pru Dividend Yie  ICICI Pru Focused Equi  Parag Parikh Flexi Cap  HDFC Flexi Cap Dir Gr  Nippon India Small Cap  Quant ESG Integration   Bandhan Large & Mid Ca  HDFC Focused Dir Gr  ICICI Pru FMCG Dir Gr  ICICI Prudential Large  Edelweiss Mid Cap Dir   Quant Flexi Cap Dir Gr  Bank of India Manfactr  Tata India Consumer Di  SBI Banking & Financia  Nippon India Growth Mi  Nippon India Multi Cap  Invesco India largecap  HDFC Mid Cap Dir Gr  Mirae Asset Great Cons  Motilal Oswal ELSS Tax  Franklin Build India D  Invesco India Mid Cap   ICICI Pru Large & Mid   Kotak India Growth Ser  ICICI Pru Infrastructu  Bank of India Flexi Ca  Mahindra Manulife Mult  Invesco India large& m  DSP Nat Res & New Enrg  Sundaram LT Tax Ad Sr   DSP India T.I.G.E.R. D  Kotak Contra Dir Gr  Sundaram LT Tax Ad Sr   Edelwei

In [29]:
def optimize_portfolio_fast(ranked_df: pd.DataFrame,
                           overlap_matrix: pd.DataFrame,
                           num_funds: int   = NUM_FUNDS,
                           max_overlap: float = MAX_OVERLAP_PCT,
                           weight_floor: float = WEIGHT_FLOOR,
                           weight_cap: float   = WEIGHT_CAP) -> dict:
    """
    Fast portfolio optimizer using:
    - Backtracking with early pruning
    - Branch & bound
    - Precomputed overlap lookup
    """

    # ── Setup ────────────────────────────────────────────────────────────────
    isins = ranked_df['ISIN'].tolist()
    score_map = dict(zip(ranked_df['ISIN'], ranked_df['score']))
    k = min(num_funds, len(isins))

    # Sort by score descending (important for pruning efficiency)
    isins = sorted(isins, key=lambda x: score_map[x], reverse=True)

    # Precompute overlap lookup (faster than pandas)
    overlap_lookup = overlap_matrix.to_dict()

    best_combo = None
    best_score = -float('inf')

    # Precompute prefix max scores for branch & bound
    scores_sorted = [score_map[i] for i in isins]

    # ── Recursive Search ─────────────────────────────────────────────────────
    def search(current_combo, start_idx, current_score):
        nonlocal best_combo, best_score

        # ✅ If full portfolio built
        if len(current_combo) == k:
            if current_score > best_score:
                best_score = current_score
                best_combo = current_combo.copy()
                logger.info(f'New best: {best_score:.2f}')
            return

        remaining_slots = k - len(current_combo)

        # ❌ Branch & Bound: max possible future score
        if start_idx + remaining_slots > len(isins):
            return

        max_possible = current_score + sum(scores_sorted[start_idx:start_idx + remaining_slots])
        if max_possible <= best_score:
            return  # prune branch

        # ── Try candidates ────────────────────────────────────────────────────
        for i in range(start_idx, len(isins)):
            candidate = isins[i]

            # ❌ Early overlap pruning
            if any(overlap_lookup[candidate][existing] > max_overlap for existing in current_combo):
                continue

            # Add candidate
            current_combo.append(candidate)

            search(
                current_combo,
                i + 1,
                current_score + score_map[candidate]
            )

            # Backtrack
            current_combo.pop()

    # ── Run Search ───────────────────────────────────────────────────────────
    search([], 0, 0.0)

    # ── Handle failure (same logic as before) ─────────────────────────────────
    effective_threshold = max_overlap

    if best_combo is None:
        for relaxed in range(int(max_overlap) + 5, int(max_overlap * 2) + 1, 5):
            logger.warning(f'No valid portfolio at ≤ {effective_threshold}%. Relaxing to {relaxed}%')

            best_combo = None
            best_score = -float('inf')

            def search_relaxed(current_combo, start_idx, current_score):
                nonlocal best_combo, best_score

                if len(current_combo) == k:
                    if current_score > best_score:
                        best_score = current_score
                        best_combo = current_combo.copy()
                    return

                remaining_slots = k - len(current_combo)
                if start_idx + remaining_slots > len(isins):
                    return

                max_possible = current_score + sum(scores_sorted[start_idx:start_idx + remaining_slots])
                if max_possible <= best_score:
                    return

                for i in range(start_idx, len(isins)):
                    candidate = isins[i]

                    if any(overlap_lookup[candidate][existing] > relaxed for existing in current_combo):
                        continue

                    current_combo.append(candidate)
                    search_relaxed(current_combo, i + 1, current_score + score_map[candidate])
                    current_combo.pop()

            search_relaxed([], 0, 0.0)
            effective_threshold = relaxed

            if best_combo is not None:
                break

    if best_combo is None:
        raise ValueError(
            f'No valid portfolio found even at {effective_threshold:.0f}% overlap.'
        )

    # ── Weight Allocation (unchanged) ─────────────────────────────────────────
    raw = {isin: max(score_map.get(isin, 0.0), 0.0) for isin in best_combo}
    total = sum(raw.values())

    initial = (
        {k: v / total for k, v in raw.items()}
        if total > 0
        else {isin: 1.0 / len(best_combo) for isin in best_combo}
    )

    weights = clip_and_renormalize_weights(initial, weight_floor, weight_cap)

    return {
        'selected_funds': list(best_combo),
        'weights': weights,
        'portfolio_score': best_score,
        'effective_overlap_pct': effective_threshold,
    }

In [30]:
# ── Step 4: Portfolio Optimization ───────────────────────────────────────────
print('=' * 70)
print('STEP 4 — Optimization')
print('=' * 70)

portfolio = optimize_portfolio_fast(
    ranked_df, overlap_df,
    num_funds    = NUM_FUNDS,
    max_overlap  = MAX_OVERLAP_PCT,
    weight_floor = WEIGHT_FLOOR,
    weight_cap   = WEIGHT_CAP,
)

if portfolio['effective_overlap_pct'] > MAX_OVERLAP_PCT:
    logger.warning(
        f"Overlap constraint relaxed to {portfolio['effective_overlap_pct']:.0f}% "
        f"(target {MAX_OVERLAP_PCT:.0f}%). Consider expanding the fund universe."
    )

INFO — New best: 571.61


STEP 4 — Optimization


## 8. Results

In [31]:
print('=' * 70)
print(f'FINAL PORTFOLIO  |  {RISK_PROFILE.upper()}  |  {NUM_FUNDS} FUNDS')
print('=' * 70)
print(f'Portfolio score    : {portfolio["portfolio_score"]:.2f}')
print(f'Overlap constraint : {portfolio["effective_overlap_pct"]:.0f}%')
print(f'Weight bounds      : [{WEIGHT_FLOOR*100:.1f}%, {WEIGHT_CAP*100:.1f}%]')
print()


def _fmt(v, fmt='5.1f'):
    """Format a score value; returns ' N/A' if None."""
    return format(v, fmt) if v is not None else f"{'N/A':>5}"


header = f'{"Wt%":>6}  {"Tier":>4}  {"Hybrid":>6}  {"Abs":>5}  {"Risk":>5}  '\
         f'{"S3Y":>5}  {"So3Y":>5}  {"S5Y":>5}  Fund'
print(header)
print('-' * len(header))

result_rows = []
for isin, wt in sorted(portfolio['weights'].items(), key=lambda x: -x[1]):
    r    = ranked_df[ranked_df['ISIN'] == isin].iloc[0]
    name = get_fund_name(isin)
    print(
        f"{wt*100:5.1f}%  "
        f"{str(r['tier']):>4}  "
        f"{r['score']:6.2f}  "
        f"{r['abs_score']:5.1f}  "
        f"{r['risk_score']:5.1f}  "
        f"{_fmt(r['sharpe_3y_score'])}  "
        f"{_fmt(r['sortino_score'])}  "
        f"{_fmt(r['sharpe_5y_score'])}  "
        f"{name}"
    )
    result_rows.append({
        'isin':            isin,
        'fund_name':       name,
        'weight_pct':      round(wt * 100, 2),
        'tier':            r['tier'],
        'hybrid_score':    r['score'],
        'abs_score':       r['abs_score'],
        'risk_score':      r['risk_score'],
        'sharpe_3y_score': r['sharpe_3y_score'],
        'sortino_score':   r['sortino_score'],
        'sharpe_5y_score': r['sharpe_5y_score'],
        'sharpe_3y':       r['sharpe_3y'],
        'sortino_3y':      r['sortino_3y'],
        'sharpe_5y':       r['sharpe_5y'],
    })

total_wt = sum(portfolio['weights'].values())
print('-' * len(header))
print(f"{total_wt*100:5.1f}%  {'':>4}  {'':>6}  {'':>5}  {'':>5}  TOTAL")

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = OUTPUT_DIR / 'portfolio.tsv'
(
    pd.DataFrame(result_rows)
    .sort_values('weight_pct', ascending=False)
    .reset_index(drop=True)
    .to_csv(out_path, sep='\t', index=False)
)
logger.info(f'Portfolio saved → {out_path}')
print(f'\n✓ Saved to {out_path}')


INFO — Portfolio saved → tuning_results/portfolio_debug/portfolio.tsv


FINAL PORTFOLIO  |  AGGRESSIVE  |  8 FUNDS
Portfolio score    : 571.61
Overlap constraint : 40%
Weight bounds      : [6.2%, 31.2%]

   Wt%  Tier  Hybrid    Abs   Risk    S3Y   So3Y    S5Y  Fund
-------------------------------------------------------------
 14.7%     A   84.17  102.0   67.6   61.9   64.3   76.0  Nippon India Large Cap Dir Gr
 14.1%     A   80.76  103.6   45.0   43.8   42.7   48.2  HSBC Value Dir Gr
 12.6%     A   71.99   93.4  100.0  100.0  100.0  100.0  SBI Technology Opportunities Dir Gr
 12.3%     A   70.08   96.4   71.5   74.7   77.9   62.8  Invesco India Financial Serv Dir Gr
 12.0%     A   68.87   99.0   47.1   40.6   43.6   56.6  Quant ELSS Tax Saver Dir Gr
 11.6%     A   66.35   95.8   65.5   62.6   65.6   68.3  ICICI Pru Focused Equity Dir Gr
 11.5%     A   65.65   95.1   68.8   64.3   77.5   65.8  Parag Parikh Flexi Cap Dir Gr
 11.2%     A   63.74   97.6   43.7   40.9   42.6   47.5  Nippon India Small Cap Dir Gr
------------------------------------------------